In [132]:
import matplotlib.pyplot as plt
import numpy as np
from obspy.imaging.beachball import aux_plane
import pandas as pd
from pathlib import Path

import sys
sys.path.append("../")
from src.synthetic_catalog_creation import synthetic_catalog_creation
from src.catalog_analysis import catalog_to_stress_and_instability
from src.instability_frequency import random_normals
from src.instability_frequency import instability_PDF
from src.plot import plot_unitmohr_with_coulomb
from src.plot import plot_instability_PDF

In [133]:
output_dir = Path("../figures/instability_frequency")
output_dir.mkdir(parents=True, exist_ok=True)

In [134]:
### input parameters ###
sigma_model = np.array([[1,0,0], [0,-1,0], [0,0,-0.8]])
Mmin = -2
Mc = 2
friction_model = 0.3
focmech_uncertainty = np.array([15, 34])    # degree

stressinverse_dir = "../../Stressinverse_1.1.3"

# relationship between instability and b-value
def model_i_to_b(instability, model_type):
    if model_type == "exp":
        return 1.1 - 0.4 * np.exp((instability - 1) / 0.4)
    if model_type == "linear":
        return np.where(
            instability >= 0.9,
            4 * (1 - instability) + 0.7,
            1.1
        )

In [135]:
columns = ["magnitude", "strike", "dip", "rake", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]

catalog_Mmin, catalog_Mc = [
    pd.DataFrame(columns=columns)
    for _ in range(2)
]

In [136]:
### catalog creation (Mmin) ###
#####
# true catalog
#####
## distribution of the stress states on Mohr circle 
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                Mmin,
                                                Mmin,
                                                friction_model,
                                                N_earthquake=10**5,
                                                N_subgroup=10**5
                                                )

normal_stress, shear_stress, instability1 = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_orig, dip_orig, rake_orig
)

fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction_model)

fig.savefig(output_dir/f"mohr_Mmin={Mmin}_true.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability (Mmin)
fig, ax = plt.subplots(constrained_layout=True)
density, bins = np.histogram(instability1, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_Mmin={Mmin}_true.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [137]:
### catalog creation (Mmin) ###
#####
# estimated catalog
#####
## distribution of the stress states on Mohr circle 
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                Mmin,
                                                Mmin,
                                                friction_model,
                                                N_earthquake=10**5,
                                                N_subgroup=10**5
                                                )
n_events = len(strike_orig)

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_orig, dip_orig, rake_orig
)
strike_aux, dip_aux, rake_aux = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)

for k in range(n_events):
    strike_aux[k], dip_aux[k], rake_aux[k] = aux_plane(strike_orig[k], dip_orig[k], rake_orig[k])
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_aux, dip_aux, rake_aux
)

mask = instability1 > instability2
normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction_model)
fig.savefig(output_dir/f"mohr_Mmin={Mmin}_est.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability (Mmin)
fig, ax = plt.subplots(constrained_layout=True)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_Mmin={Mmin}_est.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [138]:
### catalog creation (Mc) *same as catalog_no_error_exp ###
#####
# true catalog
#####
## distribution of the stress states on Mohr circle 
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                Mmin,
                                                Mc,
                                                friction_model,
                                                N_earthquake=10**5,
                                                N_subgroup=10**7
                                                )

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_orig, dip_orig, rake_orig
)

fig, ax = plot_unitmohr_with_coulomb(normal_stress1, shear_stress1, friction_model)
fig.savefig(output_dir/f"mohr_Mc={Mc}_true.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability (Mmin)
fig, ax = plt.subplots(constrained_layout=True)
density, bins = np.histogram(instability1, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_Mc={Mc}_true.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [139]:
### catalog creation (Mc) ###
#####
# estimated catalog
#####
## distribution of the stress states on Mohr circle 
n_events = len(strike_orig)
strike_aux, dip_aux, rake_aux = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)

for k in range(n_events):
    strike_aux[k], dip_aux[k], rake_aux[k] = aux_plane(strike_orig[k], dip_orig[k], rake_orig[k])
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_aux, dip_aux, rake_aux
)

mask = instability1 > instability2
normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction_model)
fig.savefig(output_dir/f"mohr_Mc={Mc}_est.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability (Mmin)
fig, ax = plt.subplots(constrained_layout=True)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_Mc={Mc}_est.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [ ]:
### Dataset1 ###
df = pd.read_csv("../data/california/processed/dataset1.csv")
sigma = df[
        ["Snn", "Sne", "Snd",
         "Sne", "See", "Sed",
         "Snd", "Sed", "Sdd"]
    ].to_numpy().reshape(-1, 3, 3)
friction = 0.32     # MLE value
strike1 = df["strike"].to_numpy()
dip1 = df["dip"].to_numpy()
rake1 = df["rake"].to_numpy()

n_events = len(strike1)
strike2, dip2, rake2 = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)
for k in range(n_events):
    strike2[k], dip2[k], rake2[k] = aux_plane(strike1[k], dip1[k], rake1[k])

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma, friction, strike1, dip1, rake1
)
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma, friction, strike2, dip2, rake2
)
mask = instability1 > instability2
normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

## distribution of stress states on Mohr circle
fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction)
fig.savefig(output_dir/f"mohr_dataset1.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability
fig, ax = plt.subplots(constrained_layout=True)
instability = np.array(instability)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_dataset1.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [ ]:
### Dataset2 ###
df = pd.read_csv("../data/california/processed/dataset2.csv")
sigma = df[
        ["Snn", "Sne", "Snd",
         "Sne", "See", "Sed",
         "Snd", "Sed", "Sdd"]
    ].to_numpy().reshape(-1, 3, 3)
friction = 0.35     # MLE value
strike1 = df["strike"].to_numpy()
dip1 = df["dip"].to_numpy()
rake1 = df["rake"].to_numpy()

n_events = len(strike1)
strike2, dip2, rake2 = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)
for k in range(n_events):
    strike2[k], dip2[k], rake2[k] = aux_plane(strike1[k], dip1[k], rake1[k])

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma, friction, strike1, dip1, rake1
)
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma, friction, strike2, dip2, rake2
)
mask = instability1 > instability2

normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

## distribution of stress states on Mohr circle
fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction)
fig.savefig(output_dir/f"mohr_dataset2.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability
fig, ax = plt.subplots(constrained_layout=True)
instability = np.array(instability)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.1, 5.1)
fig.savefig(output_dir/f"instability_PDF_dataset2.svg", bbox_inches="tight", dpi=300)
plt.close(fig)